In [1]:
import torch
import pickle
import numpy as np

In [8]:
with open(r"C:\Users\Neela\Documents\GitHub\EntityAspectLinking\picklefiles\final_eal_random.pkl", 'rb') as eal:
    data = pickle.load(eal)
    
ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [3]:
PTH = r"C:\Users\Neela\Documents\GitHub\EntityAspectLinking\picklefiles"

In [4]:
def read_tensor(filename):
    with open(f'{PTH}\\{filename}', 'rb') as f:
        emb = np.load(f, allow_pickle = True)
    return emb

In [5]:
ent_emb = read_tensor('random_targetentemb.pkl')

asp_emb = read_tensor('random_aspemb.pkl')

context_emb = read_tensor('random_paragraphemb.pkl')

In [11]:
aspect = [el[1] for el in data]
lngth = [len(el['candidate_aspects']) - 1 for el in aspect]
asp_count = sum([len(el['candidate_aspects']) for el in aspect])

In [10]:
asp_count

16383

In [16]:
import urllib
def decode(str):
    return urllib.parse.unquote(str)

In [23]:
i = 0
j = 0
check = 0
yo = 0
unid_ls = []
while(i < asp_count):
    aspect = asp[j]['true_aspect']
    aspect = decode(aspect)
    #print(j, "Entity")
    #tokens = tokenizer.tokenize(aspect)
    #print(i, 'first')
    #print("This shit", tokens)
    i+=1    
    this = 0
    
    #if len(asp[j]['candidate_aspects']) == 0:
        #print("Ache", j)
    for cand in asp[j]['candidate_aspects']:
        if i >= asp_count:
            #print("Yes ekhane")
            break
        if cand['aspect_name'] == aspect:
            check += 1
        if cand['aspect_name'] != aspect:
            this += 1
            casp = cand['aspect_name']
            #print(cand['aspect_name'], aspect)
            #tokens = tokenizer.tokenize(casp)
            #print(tokens)
            #print(i, "second")
            i+=1
    if this == len(asp[j]['candidate_aspects']):
        yo += 1  
        unid_ls.append(j)     
        #print("Found it", aspect, j)
            
    j += 1
unid_ls

[1257, 1360]

2174

In [25]:
dataset = np.zeros((asp_count, 201 + context_emb.shape[1]))

In [26]:
context_emb.shape

torch.Size([2500, 384])

In [27]:
i = 0
k = 0
for _ in range(len(dataset)):
    while(k < len(lngth)):
        if i in unid_ls:
            i += 1
            continue
        dataset[i, : ent_emb.shape[1]] = ent_emb[k]
        dataset[i, ent_emb.shape[1] : ent_emb.shape[1] + context_emb.shape[1]] = context_emb[k]
        dataset[i, ent_emb.shape[1] + context_emb.shape[1] : -1] = asp_emb[i]
        dataset[i, -1] = 1
        i += 1
        for _ in range(lngth[k]):
            dataset[i, : ent_emb.shape[1]] = ent_emb[k]
            dataset[i, ent_emb.shape[1] : ent_emb.shape[1] + context_emb.shape[1]] = context_emb[k]
            dataset[i, ent_emb.shape[1] + context_emb.shape[1] : -1] = asp_emb[i]
            dataset[i, -1] = 0
            i += 1
        k += 1

In [28]:
from imblearn.under_sampling import RandomUnderSampler

In [29]:
from sklearn.preprocessing import StandardScaler

In [30]:
X = dataset[:,:-1]
sc = StandardScaler()
X = sc.fit_transform(X)
y = dataset[:,-1]

In [31]:
undersample = RandomUnderSampler(sampling_strategy='majority')

In [35]:
X, y = undersample.fit_resample(X, y)
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.3, random_state = 42)

In [36]:
from lazypredict.Supervised import LazyClassifier
clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
models,predictions = clf.fit(X_train, X_test, y_train, y_test)

print(models)

100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [01:52<00:00,  3.88s/it]

                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
LGBMClassifier                     0.68               0.68     0.68      0.68   
XGBClassifier                      0.67               0.67     0.67      0.67   
BaggingClassifier                  0.66               0.65     0.65      0.65   
RandomForestClassifier             0.63               0.63     0.63      0.63   
AdaBoostClassifier                 0.63               0.63     0.63      0.63   
DecisionTreeClassifier             0.61               0.61     0.61      0.61   
ExtraTreesClassifier               0.60               0.60     0.60      0.60   
SVC                                0.58               0.58     0.58      0.58   
NuSVC                              0.58               0.58     0.58      0.58   
LinearSVC                          0.57               0.57     0.57      0.57   
KNeighborsClassifier        

In [43]:
y_test.shape

(1500,)

In [44]:
models,predictions = clf.fit(X_train, X_test, y_train, y_test)

print(models)

NameError: name 'clf' is not defined

In [ ]:
y_test.shape
y_pred.shape

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(y_test,y_pred)